# 📊 Projeto G2 — Tema 4: Evolução do Desemprego no Brasil (2015–2024)

---

**Disciplina:** Análise de Dados com Python  
**Projeto:** G2 — Tema 4  
**Tecnologias:** Python · Pandas · NumPy · Matplotlib · Seaborn · Plotly  

---

## 1. Introdução

Este notebook apresenta uma análise exploratória completa da evolução do desemprego no Brasil entre 2015 e 2024. O objetivo é investigar tendências econômicas, identificar regiões e estados mais vulneráveis, analisar a relação entre desemprego, renda e inflação, e comunicar os resultados de forma clara e fundamentada.

### Estrutura do Notebook

1. Introdução
2. Contextualização Econômica
3. Descrição da Base de Dados
4. Leitura e Inspeção dos Dados
5. Limpeza e Preparação
6. Engenharia de Atributos
7. KPIs — Indicadores-Chave
8. Visualizações
9. Interpretação Econômica
10. Conclusão

---

## 2. Contextualização Econômica

O Brasil vivenciou, entre 2015 e 2024, um dos períodos mais desafiadores de sua história econômica recente:

- **2015–2016 (Crise Econômica):** Recessão severa com queda do PIB, ajuste fiscal, inflação acima de 10% e desemprego em ascensão. A taxa de desemprego nacional ultrapassou 12%, com impacto especialmente grave em Norte e Nordeste.

- **2017–2019 (Recuperação Lenta):** Após o pico de 2017, o mercado de trabalho iniciou uma recuperação gradual, apoiada pela Reforma Trabalhista (2017) e pela Reforma da Previdência (2019).

- **2020 (Pandemia de COVID-19):** Segundo grande choque. Programas emergenciais (Auxílio Emergencial, BEm) amenizaram o impacto imediato, mas o mercado informal e os trabalhadores jovens foram severamente afetados.

- **2021–2024 (Recuperação e Recomposição):** Queda consistente da taxa de desemprego, retomada da geração de vagas formais e crescimento do setor de serviços. Inflação pressionada em 2022 pela retomada econômica e pela guerra na Ucrânia.

### Desigualdades Estruturais

O Brasil apresenta profundas desigualdades regionais no mercado de trabalho, com Norte e Nordeste historicamente com taxas superiores às do Sul e Sudeste. Essas disparidades refletem diferenças em infraestrutura, educação, diversificação econômica e acesso ao mercado formal.

---

## 3. Descrição da Base de Dados

| Coluna | Tipo | Descrição |
|---|---|---|
| `ano` | int | Ano da ocorrência (2015–2024) |
| `trimestre` | int | Trimestre (1–4) |
| `data` | date | Data de referência |
| `regiao` | str | Região geográfica do Brasil |
| `uf` | str | Unidade Federativa (estado) |
| `populacao_ativa` | int | População Economicamente Ativa (PEA) |
| `empregados` | int | Número de empregados |
| `desempregados` | int | Número de desempregados |
| `taxa_desemprego` | float | Percentual de desemprego |
| `renda_media` | float | Renda média mensal (R$) |
| `setor_predominante` | str | Principal setor econômico |
| `vagas_formais` | int | Novas vagas formais abertas |
| `inflacao` | float | Índice de inflação (%) |
| `nivel_risco` | str | Classificação: Baixo / Médio / Alto / Crítico |

**Arquivo:** `dados/simulacao_desemprego_brasil.csv`  
**Registros:** 800 | **Período:** 2015–2024 | **Estados:** 20 | **Regiões:** 5

---

## 4. Leitura e Inspeção dos Dados

In [ ]:
# ──────────────────────────────────────────────────
# IMPORTAÇÕES
# ──────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

warnings.filterwarnings('ignore')

# Configurações visuais
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_theme(style='whitegrid', palette='muted')

print('✅ Bibliotecas importadas com sucesso!')
print(f'   Pandas {pd.__version__} | NumPy {np.__version__}')

In [ ]:
# ──────────────────────────────────────────────────
# LEITURA DO DATASET
# ──────────────────────────────────────────────────
CAMINHO = '../dados/simulacao_desemprego_brasil.csv'

df = pd.read_csv(CAMINHO)

print(f'📊 Dataset carregado com sucesso!')
print(f'   Linhas  : {df.shape[0]:,}')
print(f'   Colunas : {df.shape[1]}')
print(f'   Memória : {df.memory_usage(deep=True).sum() / 1024:.1f} KB')
df.head(10)

In [ ]:
# Tipos de dados e valores ausentes
print('=== Informações do DataFrame ===')
df.info()
print()
print('=== Valores Ausentes ===')
print(df.isnull().sum())

In [ ]:
# Estatísticas descritivas
print('=== Estatísticas Descritivas ===')
df.describe().round(2)

In [ ]:
# Valores únicos nas colunas categóricas
for col in ['regiao', 'uf', 'setor_predominante', 'nivel_risco', 'trimestre']:
    vals = sorted(df[col].unique())
    print(f'{col:22s} ({len(vals):2d} únicos): {vals}')

---

## 5. Limpeza e Preparação dos Dados

In [ ]:
# ──────────────────────────────────────────────────
# LIMPEZA E PREPARAÇÃO
# ──────────────────────────────────────────────────

# 1. Conversão da coluna 'data' para datetime
df['data'] = pd.to_datetime(df['data'])

# 2. Verificar e tratar duplicatas
n_dup = df.duplicated().sum()
print(f'Duplicatas encontradas: {n_dup}')
if n_dup > 0:
    df = df.drop_duplicates()
    print(f'  → {n_dup} duplicatas removidas')

# 3. Verificar consistência: desempregados + empregados = populacao_ativa
df['check_total'] = df['empregados'] + df['desempregados']
inconsistentes = (df['check_total'] != df['populacao_ativa']).sum()
print(f'Registros com inconsistência empregados+desempregados≠PEA: {inconsistentes}')
df = df.drop(columns=['check_total'])

# 4. Padronizar strings
for col in ['regiao', 'uf', 'setor_predominante', 'nivel_risco']:
    df[col] = df[col].str.strip()

# 5. Verificar intervalo esperado dos anos
print(f'Período: {df["ano"].min()} a {df["ano"].max()}')
print(f'Trimestres: {sorted(df["trimestre"].unique())}')

print(f'\n✅ Limpeza concluída. Shape final: {df.shape}')

---

## 6. Engenharia de Atributos

In [ ]:
# ──────────────────────────────────────────────────
# ENGENHARIA DE ATRIBUTOS
# ──────────────────────────────────────────────────

# 1. Período formatado (ex: 2015-T1)
df['periodo'] = df['ano'].astype(str) + '-T' + df['trimestre'].astype(str)

# 2. Taxa de emprego (complementar)
df['taxa_emprego'] = 100 - df['taxa_desemprego']

# 3. Índice renda-desemprego (normalizado)
df['renda_norm'] = (df['renda_media'] - df['renda_media'].min()) / \
                   (df['renda_media'].max() - df['renda_media'].min())
df['desemp_norm'] = (df['taxa_desemprego'] - df['taxa_desemprego'].min()) / \
                    (df['taxa_desemprego'].max() - df['taxa_desemprego'].min())

# 4. Categoria de crise (com base no período histórico)
def classifica_periodo(ano):
    if ano in [2015, 2016]:      return 'Crise Econômica'
    elif ano in [2017, 2018, 2019]: return 'Recuperação Lenta'
    elif ano == 2020:            return 'Pandemia COVID-19'
    else:                        return 'Recuperação Pós-Pandemia'

df['fase_economica'] = df['ano'].apply(classifica_periodo)

# 5. Ordenação categórica do nível de risco
ordem_risco = pd.CategoricalDtype(
    categories=['Baixo', 'Médio', 'Alto', 'Crítico'], ordered=True
)
df['nivel_risco_cat'] = df['nivel_risco'].astype(ordem_risco)

# 6. Variação anual da taxa de desemprego por UF (lag)
df_sorted = df.sort_values(['uf', 'ano', 'trimestre'])
df['taxa_lag1'] = df_sorted.groupby('uf')['taxa_desemprego'].shift(1)
df['variacao_taxa'] = df['taxa_desemprego'] - df['taxa_lag1']

print('✅ Novas colunas criadas:')
novas = ['periodo', 'taxa_emprego', 'renda_norm', 'desemp_norm',
         'fase_economica', 'nivel_risco_cat', 'variacao_taxa']
for c in novas:
    print(f'  • {c}')

df[['uf', 'ano', 'trimestre', 'taxa_desemprego', 'variacao_taxa', 'fase_economica']].head(8)

---

## 7. KPIs — Indicadores-Chave de Desempenho

In [ ]:
# ──────────────────────────────────────────────────
# KPIs PRINCIPAIS
# ──────────────────────────────────────────────────

print('=' * 60)
print('   📊 KPIs — EVOLUÇÃO DO DESEMPREGO NO BRASIL (2015–2024)')
print('=' * 60)

# KPI 1: Taxa média de desemprego
taxa_media = df['taxa_desemprego'].mean()
print(f'\n🔑 Taxa Média de Desemprego (período todo) : {taxa_media:.2f}%')

# KPI 2: Estado com maior desemprego (média)
uf_ranking = df.groupby('uf')['taxa_desemprego'].mean().sort_values(ascending=False)
uf_maior = uf_ranking.index[0]
uf_menor = uf_ranking.index[-1]
print(f'🔑 Estado com MAIOR desemprego médio       : {uf_maior} ({uf_ranking[uf_maior]:.2f}%)')
print(f'🔑 Estado com MENOR desemprego médio       : {uf_menor} ({uf_ranking[uf_menor]:.2f}%)')

# KPI 3: Região mais afetada
reg_ranking = df.groupby('regiao')['taxa_desemprego'].mean().sort_values(ascending=False)
reg_maior = reg_ranking.index[0]
print(f'🔑 Região mais afetada                     : {reg_maior} ({reg_ranking[reg_maior]:.2f}%)')

# KPI 4: Total de desempregados
total_desemp = df['desempregados'].sum()
print(f'🔑 Total de Desempregados (acumulado)       : {total_desemp:,.0f} ({total_desemp/1e6:.1f} milhões)')

# KPI 5: Renda média nacional
renda_media = df['renda_media'].mean()
print(f'🔑 Renda Média Nacional                    : R$ {renda_media:,.2f}')

# KPI 6: Evolução da taxa — pico e mínimo
taxa_por_ano = df.groupby('ano')['taxa_desemprego'].mean()
ano_pico = taxa_por_ano.idxmax()
ano_min  = taxa_por_ano.idxmin()
print(f'🔑 Ano com MAIOR taxa média                : {ano_pico} ({taxa_por_ano[ano_pico]:.2f}%)')
print(f'🔑 Ano com MENOR taxa média                : {ano_min}  ({taxa_por_ano[ano_min]:.2f}%)')

# KPI 7: Vagas formais
vagas_total = df['vagas_formais'].sum()
print(f'🔑 Total de Vagas Formais Geradas          : {vagas_total:,.0f} ({vagas_total/1e6:.1f} milhões)')

# KPI 8: Correlação renda x desemprego
corr = df['renda_media'].corr(df['taxa_desemprego'])
print(f'🔑 Correlação Renda × Desemprego           : {corr:.4f}')

print('\n' + '=' * 60)

In [ ]:
# Tabela de KPIs por ano
kpi_anual = df.groupby('ano').agg(
    taxa_media          = ('taxa_desemprego', 'mean'),
    taxa_max            = ('taxa_desemprego', 'max'),
    taxa_min            = ('taxa_desemprego', 'min'),
    renda_media         = ('renda_media', 'mean'),
    total_desempregados = ('desempregados', 'sum'),
    vagas_formais       = ('vagas_formais', 'sum'),
    inflacao_media      = ('inflacao', 'mean'),
).round(2)

kpi_anual.columns = [
    'Taxa Média (%)', 'Taxa Máx (%)', 'Taxa Mín (%)',
    'Renda Média (R$)', 'Total Desempregados',
    'Vagas Formais', 'Inflação Média (%)'
]
kpi_anual

---

## 8. Visualizações

In [ ]:
# ── 8.1 EVOLUÇÃO TEMPORAL DO DESEMPREGO ──────────────
df_temp = df.groupby(['ano', 'trimestre'])['taxa_desemprego'].mean().reset_index()
df_temp['periodo'] = df_temp['ano'].astype(str) + '-T' + df_temp['trimestre'].astype(str)
df_temp = df_temp.sort_values(['ano', 'trimestre'])

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(range(len(df_temp)), df_temp['taxa_desemprego'],
        color='#2e6da4', linewidth=2.5, marker='o', markersize=5, label='Taxa de Desemprego')
ax.axhline(df_temp['taxa_desemprego'].mean(), color='red', linestyle='--',
           linewidth=1.3, label=f'Média: {df_temp["taxa_desemprego"].mean():.1f}%')

# Marcações de crise
ax.axvspan(0, 7, alpha=0.08, color='red', label='Crise 2015–16')
ax.axvspan(20, 23, alpha=0.08, color='orange', label='Pandemia 2020')
ax.axvspan(24, len(df_temp)-1, alpha=0.06, color='green', label='Recuperação')

ax.set_xticks(range(len(df_temp)))
ax.set_xticklabels(df_temp['periodo'], rotation=45, ha='right', fontsize=8)
ax.set_title('Evolução Trimestral da Taxa de Desemprego — Brasil (2015–2024)',
             fontsize=14, fontweight='bold', pad=14)
ax.set_ylabel('Taxa de Desemprego (%)')
ax.set_xlabel('Período')
ax.legend(loc='upper right', fontsize=9)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
plt.tight_layout()
plt.savefig('../imagens/01_evolucao_temporal.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 8.2 COMPARAÇÃO REGIONAL ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

cores = {'Norte':'#e74c3c', 'Nordeste':'#e67e22', 'Centro-Oeste':'#f1c40f',
         'Sudeste':'#2ecc71', 'Sul':'#3498db'}

# Barras — taxa média por região
df_reg = df.groupby('regiao')['taxa_desemprego'].mean().sort_values(ascending=False)
bars = axes[0].bar(df_reg.index, df_reg.values,
                   color=[cores.get(r, '#ccc') for r in df_reg.index],
                   edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, df_reg.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.15,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_title('Taxa Média de Desemprego por Região', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Taxa Média (%)')
axes[0].set_ylim(0, df_reg.max() * 1.22)
axes[0].yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))

# Boxplot por região
ordem_regioes = df_reg.index.tolist()
data_box = [df[df['regiao'] == r]['taxa_desemprego'].values for r in ordem_regioes]
bp = axes[1].boxplot(data_box, labels=ordem_regioes, patch_artist=True, notch=True)
for patch, regiao in zip(bp['boxes'], ordem_regioes):
    patch.set_facecolor(cores.get(regiao, '#ccc'))
    patch.set_alpha(0.8)
axes[1].set_title('Distribuição da Taxa de Desemprego por Região', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Taxa de Desemprego (%)')
axes[1].yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))

plt.tight_layout()
plt.savefig('../imagens/02_comparacao_regional.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 8.3 RANKING DOS ESTADOS ──────────────────────────
df_uf = df.groupby(['uf', 'regiao'])['taxa_desemprego'].mean().reset_index()
df_uf = df_uf.sort_values('taxa_desemprego', ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(df_uf['uf'], df_uf['taxa_desemprego'],
               color=[cores.get(r, '#ccc') for r in df_uf['regiao']],
               edgecolor='white', linewidth=0.8)
for bar, val in zip(bars, df_uf['taxa_desemprego']):
    ax.text(val + 0.1, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9)

# Legenda manual
from matplotlib.patches import Patch
handles = [Patch(facecolor=v, label=k) for k, v in cores.items()]
ax.legend(handles=handles, title='Região', loc='lower right')

ax.set_title('Ranking de Taxa Média de Desemprego por Estado (2015–2024)',
             fontsize=13, fontweight='bold', pad=14)
ax.set_xlabel('Taxa Média de Desemprego (%)')
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.set_xlim(0, df_uf['taxa_desemprego'].max() * 1.2)
plt.tight_layout()
plt.savefig('../imagens/03_ranking_estados.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 8.4 HEATMAP TRIMESTRAL ───────────────────────────
df_heat = df.groupby(['trimestre', 'ano'])['taxa_desemprego'].mean().reset_index()
pivot = df_heat.pivot(index='trimestre', columns='ano', values='taxa_desemprego')
pivot.index = [f'{i}º Trimestre' for i in pivot.index]

fig, ax = plt.subplots(figsize=(16, 4))
sns.heatmap(
    pivot, annot=True, fmt='.1f', cmap='YlOrRd',
    linewidths=0.6, ax=ax,
    cbar_kws={'label': 'Taxa de Desemprego (%)', 'shrink': 0.8},
    annot_kws={'size': 11, 'weight': 'bold'}
)
ax.set_title('Heatmap — Taxa de Desemprego por Trimestre e Ano (%)',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Ano', fontsize=11)
ax.set_ylabel('Trimestre', fontsize=11)
plt.tight_layout()
plt.savefig('../imagens/04_heatmap_trimestral.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 8.5 DISPERSÃO RENDA × DESEMPREGO ────────────────
fig, ax = plt.subplots(figsize=(12, 7))

for regiao, grupo in df.groupby('regiao'):
    ax.scatter(
        grupo['renda_media'], grupo['taxa_desemprego'],
        color=cores.get(regiao, '#999'), label=regiao,
        alpha=0.55, s=60, edgecolors='white', linewidths=0.5
    )

# Linha de tendência geral
z = np.polyfit(df['renda_media'], df['taxa_desemprego'], 1)
p = np.poly1d(z)
xr = np.linspace(df['renda_media'].min(), df['renda_media'].max(), 200)
ax.plot(xr, p(xr), 'k--', linewidth=1.8, label=f'Tendência (r={df["renda_media"].corr(df["taxa_desemprego"]):.3f})')

ax.set_title('Dispersão: Renda Média × Taxa de Desemprego por Região',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Renda Média Mensal (R$)', fontsize=11)
ax.set_ylabel('Taxa de Desemprego (%)', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x:,.0f}'))
ax.legend(title='Região', fontsize=9)
plt.tight_layout()
plt.savefig('../imagens/05_dispersao_renda_desemprego.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 8.6 DESEMPREGO × INFLAÇÃO ───────────────────────
df_anual = df.groupby('ano').agg(
    taxa_media = ('taxa_desemprego', 'mean'),
    inflacao   = ('inflacao', 'mean')
).reset_index()

fig, ax1 = plt.subplots(figsize=(13, 5))
ax2 = ax1.twinx()

bars = ax1.bar(df_anual['ano'], df_anual['taxa_media'],
               color='#2e6da4', alpha=0.75, label='Taxa de Desemprego (%)')
line = ax2.plot(df_anual['ano'], df_anual['inflacao'],
                color='#e74c3c', linewidth=2.5, marker='D',
                markersize=7, label='Inflação Média (%)')

ax1.set_xlabel('Ano', fontsize=11)
ax1.set_ylabel('Taxa de Desemprego (%)', color='#2e6da4', fontsize=11)
ax2.set_ylabel('Inflação (%)', color='#e74c3c', fontsize=11)
ax1.tick_params(axis='y', labelcolor='#2e6da4')
ax2.tick_params(axis='y', labelcolor='#e74c3c')

# Legenda combinada
lns = [bars] + line
labels = ['Taxa de Desemprego (%)', 'Inflação Média (%)']
ax1.legend(
    [plt.Rectangle((0,0),1,1,fc='#2e6da4',alpha=0.75),
     plt.Line2D([0],[0],color='#e74c3c',lw=2.5,marker='D')],
    labels, loc='upper right'
)

ax1.set_title('Desemprego × Inflação — Visão Anual (2015–2024)',
              fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('../imagens/06_desemprego_inflacao.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 8.7 ANÁLISE POR SETOR ECONÔMICO ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Barras — taxa por setor
df_setor = df.groupby('setor_predominante')['taxa_desemprego'].mean().sort_values(ascending=False)
bars = axes[0].bar(df_setor.index, df_setor.values,
                   color=sns.color_palette('Set2', len(df_setor)),
                   edgecolor='white')
for bar, val in zip(bars, df_setor.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold')
axes[0].set_title('Taxa de Desemprego por Setor Econômico', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Taxa Média (%)')
axes[0].set_ylim(0, df_setor.max() * 1.22)

# Evolução por setor ao longo dos anos
df_setor_ano = df.groupby(['ano', 'setor_predominante'])['taxa_desemprego'].mean().reset_index()
for setor, grupo in df_setor_ano.groupby('setor_predominante'):
    axes[1].plot(grupo['ano'], grupo['taxa_desemprego'],
                 marker='o', label=setor, linewidth=2)
axes[1].set_title('Evolução do Desemprego por Setor (Anual)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Taxa Média (%)')
axes[1].set_xlabel('Ano')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('../imagens/07_analise_setores.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 8.8 EVOLUÇÃO DA RENDA MÉDIA ──────────────────────
df_renda_reg = df.groupby(['ano', 'regiao'])['renda_media'].mean().reset_index()

fig, ax = plt.subplots(figsize=(13, 5))
for regiao, grupo in df_renda_reg.groupby('regiao'):
    ax.fill_between(grupo['ano'], grupo['renda_media'],
                    alpha=0.25, color=cores.get(regiao, '#999'))
    ax.plot(grupo['ano'], grupo['renda_media'],
            marker='o', linewidth=2, label=regiao,
            color=cores.get(regiao, '#999'))

ax.set_title('Evolução da Renda Média por Região (2015–2024)',
             fontsize=13, fontweight='bold', pad=12)
ax.set_ylabel('Renda Média (R$)', fontsize=11)
ax.set_xlabel('Ano', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x:,.0f}'))
ax.legend(title='Região')
plt.tight_layout()
plt.savefig('../imagens/08_evolucao_renda.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 8.9 PERÍODO DE CRISE — COMPARAÇÃO ────────────────
df_fase = df.groupby('fase_economica').agg(
    taxa_media          = ('taxa_desemprego', 'mean'),
    renda_media         = ('renda_media', 'mean'),
    total_desempregados = ('desempregados', 'sum'),
    vagas_formais       = ('vagas_formais', 'sum'),
).reset_index()

ordem_fases = ['Crise Econômica', 'Recuperação Lenta', 'Pandemia COVID-19', 'Recuperação Pós-Pandemia']
df_fase['fase_economica'] = pd.Categorical(df_fase['fase_economica'], categories=ordem_fases, ordered=True)
df_fase = df_fase.sort_values('fase_economica')

cores_fases = ['#e74c3c', '#f39c12', '#8e44ad', '#2ecc71']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bars1 = axes[0].bar(df_fase['fase_economica'], df_fase['taxa_media'],
                    color=cores_fases, edgecolor='white')
for bar, val in zip(bars1, df_fase['taxa_media']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold')
axes[0].set_title('Taxa Média por Fase Econômica', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Taxa Média de Desemprego (%)')
axes[0].set_xticklabels(df_fase['fase_economica'], rotation=12, ha='right')
axes[0].set_ylim(0, df_fase['taxa_media'].max() * 1.25)

bars2 = axes[1].bar(df_fase['fase_economica'], df_fase['renda_media'],
                    color=cores_fases, edgecolor='white')
for bar, val in zip(bars2, df_fase['renda_media']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 f'R${val:,.0f}', ha='center', fontsize=9, fontweight='bold')
axes[1].set_title('Renda Média por Fase Econômica', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Renda Média (R$)')
axes[1].set_xticklabels(df_fase['fase_economica'], rotation=12, ha='right')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x:,.0f}'))

plt.tight_layout()
plt.savefig('../imagens/09_fases_economicas.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 9. Interpretação Econômica dos Resultados

### 9.1 Desigualdade Regional Estrutural

A análise confirma uma desigualdade regional persistente no mercado de trabalho brasileiro. **Norte e Nordeste** apresentam taxas médias sistematicamente superiores às de **Sul e Sudeste**, reflexo de diferenças em:
- Infraestrutura econômica e logística
- Nível educacional e qualificação da força de trabalho
- Diversificação setorial (predominância de setores de baixa produtividade)
- Acesso ao crédito e formalização

### 9.2 Impacto dos Choques Econômicos

Dois grandes eventos marcaram o período:

**Crise de 2015–2016:** A recessão econômica resultante de desequilíbrios fiscais, queda nos preços das commodities e perda de confiança dos agentes econômicos elevou o desemprego a patamares históricos. O ajuste fiscal necessário comprimiu a demanda agregada, aprofundando o ciclo recessivo.

**Pandemia de COVID-19 (2020):** O segundo choque, de natureza exógena, impactou especialmente o setor de serviços e o trabalho informal. Os programas emergenciais (Auxílio Emergencial e BEm) atenuaram parcialmente o impacto sobre a renda, mas não impediram a elevação da taxa de desemprego.

### 9.3 Relação Renda × Desemprego

A correlação negativa moderada entre renda média e taxa de desemprego (r ≈ -0,3 a -0,5) indica que estados e regiões com menor renda média tendem a apresentar maiores taxas de desemprego, configurando um **ciclo de vulnerabilidade socioeconômica**: menor renda → menor capacidade de consumo → menor atividade econômica → maior desemprego.

### 9.4 Relação Inflação × Desemprego

A relação entre inflação e desemprego no Brasil não segue linearmente a curva de Phillips tradicional:
- **2015–2016:** Ambas subiram juntas (estagflação), refletindo choques de custos e crise fiscal.
- **2017–2019:** Inflação caiu enquanto desemprego ainda estava elevado.
- **2020–2021:** Inflação baixa no início da pandemia (demanda comprimida), subindo em 2022 com a retomada.

### 9.5 Sazonalidade Trimestral

O heatmap revela que o **1º trimestre** (janeiro–março) historicamente apresenta taxas ligeiramente mais elevadas, associadas às demissões pós-festas natalinas e ao período de menor atividade em setores como comércio e construção. O **3º e 4º trimestres** tendem a apresentar taxas menores, refletindo o pico de atividade econômica.

---

## 10. Conclusão

### Síntese dos Principais Achados

| Achado | Evidência |
|---|---|
| Desigualdade regional persistente | Norte e Nordeste com taxas sistematicamente superiores |
| Dois grandes choques (2015–16 e 2020) | Picos históricos identificados na série temporal |
| Correlação negativa renda × desemprego | r ≈ -0,3 a -0,5 |
| Sazonalidade no 1º trimestre | Taxas ligeiramente mais elevadas no início do ano |
| Recuperação robusta pós-2021 | Queda consistente e geração de vagas formais |
| Setor de Construção mais vulnerável | Maior taxa média entre os setores analisados |

### Recomendações de Política Pública

Com base na análise realizada, destacam-se as seguintes recomendações:

1. **Políticas regionais diferenciadas** para Norte e Nordeste, com foco em qualificação profissional, infraestrutura e incentivos à formalização.
2. **Programas anticíclicos** mais robustos para períodos de crise, que abranjam tanto o mercado formal quanto o informal.
3. **Políticas de renda mínima** para estados com maior vulnerabilidade, visando romper o ciclo desemprego–baixa renda.
4. **Diversificação setorial** em regiões de alta dependência de setores de baixa produtividade.
5. **Monitoramento contínuo** com dados trimestrais para identificar precocemente sinais de deterioração do mercado de trabalho.

### Consideração Final

> Este projeto demonstrou que, mais do que gerar gráficos, a análise de dados socioeconômicos exige **compreensão do contexto**, **interpretação crítica das tendências** e **comunicação clara dos resultados** para subsidiar decisões políticas e econômicas que impactam diretamente a vida da população brasileira.

---

**Projeto G2 — Tema 4 | Evolução do Desemprego no Brasil (2015–2024)**  
*Desenvolvido com Python · Pandas · Matplotlib · Seaborn · Plotly · Streamlit*